<a href="https://colab.research.google.com/github/likitha888/AI-Disease-Prediction-Chatbot/blob/main/Copy_of_Health_Symptom_Checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import difflib

# Load dataset
df = pd.read_csv("/content/cap_dataset_expanded_cleaned (1).csv")

# Combine all symptom columns into a list
symptom_cols = [col for col in df.columns if col.startswith("Symptom_")]
df['Symptoms'] = df[symptom_cols].values.tolist()

# Clean and standardize symptoms
df['Symptoms'] = df['Symptoms'].apply(lambda x: [sym.strip().lower() for sym in x if isinstance(sym, str)])

# One-hot encode symptoms
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(df['Symptoms'])
y = df['Disease']

# Split into train and test for accuracy evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate accuracy
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy*100:.2f}%")
# Map readable symptoms
symptom_name_map = {sym.lower().replace("_", " "): sym for sym in mlb.classes_}

# Fuzzy matcher for typos
def get_closest_symptom(symptom):
    candidates = list(symptom_name_map.keys())
    match = difflib.get_close_matches(symptom.lower(), candidates, n=1, cutoff=0.8)
    return symptom_name_map[match[0]] if match else None

# Predict disease from user symptoms
def predict_disease(user_symptoms):
    cleaned = []
    for sym in user_symptoms:
        key = sym.strip().lower()
        matched_sym = symptom_name_map.get(key) or get_closest_symptom(key)
        if matched_sym:
            cleaned.append(matched_sym)

    if not cleaned:
        return "No known symptoms entered. Please try again with valid symptoms."

    input_encoded = mlb.transform([cleaned])
    prediction = model.predict(input_encoded)
    return prediction[0]

# Show known symptoms
print("Known symptoms are:")
print(", ".join(sorted([sym.replace("_", " ") for sym in mlb.classes_])))

# User input loop
while True:
    user_input = input("\nEnter symptoms separated by commas (or type 'exit' to quit): ")
    if user_input.lower() == 'exit':
        print("Goodbye!")
        break

    user_symptoms = user_input.split(",")
    result = predict_disease(user_symptoms)

    print(f"\nPredicted Disease: {result}")

Model Accuracy: 70.59%
Known symptoms are:
abdominal bloating, abdominal pain, acid reflux, acidity, acne, altered sensorium, anxiety, back pain, bad breath, bald spots, blackheads, bladder discomfort, bleeding gums, blister on lips, blisters, bloating, bloody stool, blurred and distorted vision, blurred vision, body ache, breathlessness, bruising, burning, burning micturition, burning sensation, burning urination, chest pain, chest tightness, chills, chronic cough, cloudy urine, cold intolerance, cold sore, confusion, congestion, constipation, continuous feel of urine, continuous sneezing, cough, cracks in corners, cramps, crusting, dark urine, dehydration, dental caries, diarrhea, diarrhoea, difficulty eating, difficulty swallowing, discharge, dischromic  patches, distention of abdomen, dizziness, dry cough, dry eyes, dry flaky skin, dry mouth, dry skin, ear fullness, ear pain, ear pressure, extra marital contacts, eye pain, eye redness, family history, fatigue, fever, fluid discharg

In [ ]:
,